In [0]:
%pip install -r churn_model/requirements.txt --quiet --disable-pip-version-check 2>/dev/null

In [0]:
%restart_python

In [0]:
try:
    environment = dbutils.widgets.get("environment").strip().lower()
except Exception:
    environment = "dev"

TARGET_CATALOG = "company" if environment == "prd" else "company_dev"
print(f"Environment: {environment}, Catalog: {TARGET_CATALOG}")

In [0]:
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils

dbutils = DBUtils()
spark = SparkSession.builder.getOrCreate()

In [0]:
%run ./config

In [0]:
%run ./calculate_churn

In [0]:
%run ./churn_model

In [0]:
%run ./engineer_features

In [0]:
%run ./ignore_test_customer_ids

In [0]:
%run ./validate_model

In [0]:
from datetime import datetime
import logging
import os
import pytz

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{config.default.schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {TARGET_CATALOG}.{config.default.schema}.{config.default.logging_volume}")

aest = pytz.timezone('Australia/Sydney')
log_filename = datetime.now(aest).strftime("%Y-%m-%d-%H-%M-%S") + ".log"
log_path = f"/Volumes/{TARGET_CATALOG}/{config.default.schema}/{config.default.logging_volume}/{log_filename}"
local_log_path = f"/tmp/{log_filename}"
print(f"Log file: {log_path}")

log_format = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

class VolumeLogHandler(logging.Handler):
    """Appends to a local /tmp file and overwrites the Volume file with full content.
    Volumes FUSE mount does not support append/seek, so 'w' mode is used on the Volume.
    """
    def __init__(self, local_path, volume_path):
        super().__init__()
        self.local_path = local_path
        self.volume_path = volume_path

    def emit(self, record):
        msg = self.format(record) + "\n"
        with open(self.local_path, "a") as f:
            f.write(msg)
        with open(self.local_path, "r") as src:
            content = src.read()
        with open(self.volume_path, "w") as dst:
            dst.write(content)


root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
root_logger.handlers.clear()

fh = VolumeLogHandler(local_log_path, log_path)
fh.setLevel(logging.INFO)
fh.setFormatter(log_format)
root_logger.addHandler(fh)

sh = logging.StreamHandler()
sh.setLevel(logging.INFO)
sh.setFormatter(log_format)
root_logger.addHandler(sh)

logger = logging.getLogger("churn_model")
logger.info(f"Model logging started | environment={environment} | catalog={TARGET_CATALOG} | log_file={log_path}")